In [1]:
import pandas as pd

In [7]:
pd.read_csv("terms.csv")

,Criterion,Definition
0,Research topic,Broad scientific subject or problem studied.
1,Specific Claim,"Specific proposition, argument, or conclusion."
2,Method,"Method, model, algorithm, experiment, or analy..."
3,Document Structure,Where or how information appears in the document.
4,Terminology,"Remembered technical term, phrase, acronym, or..."
5,Application/Motivation,"Practical use case, motivation, or reason for ..."
6,Comparison,"Relation to another method, paper, condition, ..."
7,Exclusion Criteria,Something the target is known not to be or con...
8,Result/Finding,"Empirical, observational, or quantitative outc..."
9,Dataset,"Dataset, corpus, sample, participants, or obse..."


In [17]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

annotated_queries = pd.read_csv("annotated_queries.csv")

source_col = annotated_queries.columns[0]
url_col = annotated_queries.columns[1]
topic_cols = annotated_queries.columns[2:].tolist()

# Normalize URLs so minor formatting differences do not break pairing
annotated_queries[url_col] = (
    annotated_queries[url_col]
    .astype(str)
    .str.strip()
    .str.rstrip("/")
)

# Convert annotations to binary values
binary_annotations = annotated_queries.copy()

binary_annotations[topic_cols] = (
    binary_annotations[topic_cols]
    .apply(lambda col: col.astype(str).str.strip().str.lower().eq("x"))
    .astype(int)
)

# Every query should have exactly two annotations
annotation_counts = binary_annotations.groupby(url_col).size()

print("Number of unique queries:", annotation_counts.size)
print("Number of annotation rows:", len(binary_annotations))
print("\nAnnotations per query:")
print(annotation_counts.value_counts().sort_index())

invalid_pairs = annotation_counts[annotation_counts != 2]

if not invalid_pairs.empty:
    print("\nQueries without exactly two annotations:")
    print(invalid_pairs)

Number of unique queries: 51
Number of annotation rows: 102

Annotations per query:
2    51
Name: count, dtype: int64


In [18]:
def calculate_percentages_and_kappa(df, item_col, annotation_cols):
    percentages = (
        df[annotation_cols]
        .mean()
        .mul(100)
        .round(1)
    )

    paired = (
        df.groupby(item_col, sort=False)[annotation_cols]
        .agg(list)
    )

    kappas = {}

    for col in annotation_cols:
        pairs = paired[col]

        # Only use items with exactly two annotations
        valid_pairs = pairs[pairs.str.len().eq(2)]

        annotator_1 = valid_pairs.str[0]
        annotator_2 = valid_pairs.str[1]

        # Kappa is undefined if both annotators use only one category
        if (
            annotator_1.nunique() == 1
            and annotator_2.nunique() == 1
            and annotator_1.iloc[0] == annotator_2.iloc[0]
        ):
            kappas[col] = float("nan")
        else:
            kappas[col] = cohen_kappa_score(
                annotator_1,
                annotator_2
            )

    results = pd.DataFrame({
        "Percentage": percentages,
        "Cohen's kappa": pd.Series(kappas)
    })

    results["Cohen's kappa"] = results["Cohen's kappa"].round(3)

    return results


topic_results = calculate_percentages_and_kappa(
    binary_annotations,
    item_col=url_col,
    annotation_cols=topic_cols
)

topic_results

,Percentage,Cohen's kappa
Research topic,93.1,0.847
Specific Claim,35.3,0.744
Method,38.2,0.710
Document Structure,3.9,1.000
Terminology,19.6,0.754
Application/Motivation,4.9,0.790
Comparison,5.9,0.648
Exclusion Criteria,9.8,0.778
Result/ Finding,28.4,0.952
Dataset,6.9,0.546


In [19]:
annotated_queries = pd.read_csv("annotated_queries.csv")

# Normalize Present column
present_mask = (
    annotated_queries["Present"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("x")
)

# Keep only queries where a visual element is present
visual_queries = annotated_queries[present_mask]

# Replace this with the actual visual-element columns
visual_cols = ["Graph Type", "Structural", "Semantic Details"]

# P(element | Present)
visual_conditional_percentages = (
    visual_queries[visual_cols]
    .apply(lambda col: col.astype(str).str.strip().str.lower().eq("x").mean())
    * 100
).round(1)

visual_conditional_percentages

Graph Type          75.0
Structural          50.0
Semantic Details    56.2
dtype: float64